In [ ]:
"""
Script to build Quran embeddings for all verses and save them to
quran_embeddings.json. After running, your main app can load this file
directly without needing to recompute embeddings.

The embeddings are based on Arabic text and are language-agnostic.
The dictionary is read from the default language (German) to get Arabic keys.
"""

# ── Switch me: "openai" or "gemini" ──────────────────────────────────────────
PROVIDER = "openai"

# OpenAI config
OPENAI_EMBEDDING_MODEL = "text-embedding-3-large"
OPENAI_OUTPUT_PATH = "../data/embeddings/quran_embeddings.json"
OPENAI_BATCH_SIZE = 64

# Gemini config
GEMINI_EMBEDDING_MODEL = "gemini-embedding-001"
GEMINI_OUTPUT_PATH = "../data/embeddings/quran_embeddings_gemini.json"
GEMINI_BATCH_SIZE = 100

# Common source: Arabic verse keys come from the German Quran dictionary
QURAN_DICT_PATH = "../data/translations/quran/de.json"


In [ ]:

import os
import json
import time
from math import ceil


def load_json(path):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def save_json(path, data):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)
    os.replace(tmp, path)


# ── Resolve provider-specific settings ───────────────────────────────────────
if PROVIDER == "openai":
    OUTPUT_PATH = OPENAI_OUTPUT_PATH
    EMBEDDING_MODEL = OPENAI_EMBEDDING_MODEL
    BATCH_SIZE = OPENAI_BATCH_SIZE
elif PROVIDER == "gemini":
    OUTPUT_PATH = GEMINI_OUTPUT_PATH
    EMBEDDING_MODEL = GEMINI_EMBEDDING_MODEL
    BATCH_SIZE = GEMINI_BATCH_SIZE
else:
    raise ValueError(f'Unknown PROVIDER "{PROVIDER}" — use "openai" or "gemini".')


# ── Client setup ─────────────────────────────────────────────────────────────
if PROVIDER == "openai":
    from openai import OpenAI
    client = OpenAI()  # reads OPENAI_API_KEY env var / .env
    print("OpenAI client ready.")

else:  # gemini
    import sys
    sys.path.insert(0, "..")

    api_key = (
        os.getenv("GEMINI_API_KEY")
        or os.getenv("GOOGLE_API_KEY")
        or ""
    ).strip()

    if not api_key:
        try:
            from utils.keyring_storage import get_api_key_from_keyring
            api_key = get_api_key_from_keyring("gemini") or ""
        except Exception:
            pass

    if not api_key:
        raise RuntimeError(
            "No Gemini API key found. Set GEMINI_API_KEY / GOOGLE_API_KEY env var, "
            "or store it via the MinbarLive app (OS keychain entry 'gemini_api_key')."
        )

    from google import genai as google_genai
    client = google_genai.Client(api_key=api_key)
    print("Gemini client ready.")


# ── Load data ─────────────────────────────────────────────────────────────────
quran_dict = load_json(QURAN_DICT_PATH)
if not quran_dict:
    raise RuntimeError(f"Quran dictionary empty or not found at {QURAN_DICT_PATH}")

embeddings: dict = load_json(OUTPUT_PATH)

all_verses = list(quran_dict.keys())
verses_to_embed = [v for v in all_verses if v not in embeddings]

print(f"Provider       : {PROVIDER}  ({EMBEDDING_MODEL})")
print(f"Output         : {OUTPUT_PATH}")
print(f"Verses total   : {len(all_verses)}")
print(f"Already done   : {len(embeddings)}")
print(f"Still to embed : {len(verses_to_embed)}")

if not verses_to_embed:
    print("\nAll verses already have embeddings. Nothing to do. ✅")
    raise SystemExit(0)


In [ ]:

def embed_batch_openai(texts: list[str]) -> list[list[float]]:
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in resp.data]


def embed_batch_gemini(texts: list[str]) -> list[list[float]]:
    resp = client.models.embed_content(model=EMBEDDING_MODEL, contents=texts)
    return [list(e.values) for e in resp.embeddings]


embed_batch = embed_batch_openai if PROVIDER == "openai" else embed_batch_gemini

# ── Batch indexing loop ───────────────────────────────────────────────────────
num_batches = ceil(len(verses_to_embed) / BATCH_SIZE)
print(f"Starting indexing: {num_batches} batches (batch size {BATCH_SIZE}).")
print("You can stop anytime and restart — it resumes from where the output JSON left off.\n")

for batch_idx in range(num_batches):
    start = batch_idx * BATCH_SIZE
    batch_verses = verses_to_embed[start : start + BATCH_SIZE]
    if not batch_verses:
        continue

    print(
        f"Batch {batch_idx + 1}/{num_batches} | "
        f"Verses {start + 1}–{start + len(batch_verses)} "
        f"(saved so far: {len(embeddings)})"
    )

    try:
        t0 = time.time()
        batch_embeddings = embed_batch(batch_verses)
        elapsed = time.time() - t0
    except Exception as e:
        print(f"\nError on batch {batch_idx + 1}: {e}")
        print("Wait a moment, then re-run this cell — it will continue from here.\n")
        break

    for verse_text, emb in zip(batch_verses, batch_embeddings):
        embeddings[verse_text] = emb

    save_json(OUTPUT_PATH, embeddings)
    print(f"  Saved. Duration: {elapsed:.2f}s\n")

print("Indexing complete (or stopped at error).")
print(f"Total embeddings saved: {len(embeddings)}")
print("Done. ✅")
print(f"\nNext step: run  python notebooks/build_embeddings_npz.py  (PROVIDER = \"{PROVIDER}\")")
print("to convert this JSON into the compact .npz the app loads.")


In [ ]:

# Verify the output
final = load_json(OUTPUT_PATH)
print(f"Provider : {PROVIDER}  ({EMBEDDING_MODEL})")
print(f"File     : {OUTPUT_PATH}")
print(f"Total    : {len(final)} embeddings")
print(f"Dims     : {len(next(iter(final.values())))} per verse")
print(f"Sample   : {list(final.keys())[:3]}")
